In [ ]:
import sympy
import control as con
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from feed_conc_ctrl.plot_utils import make_tsplots

## PI Regulator in Interactive Form

$$
G_c(s) = K_c \frac{T_i s + 1}{T_i s}
$$                                                                                
   
where:                                                                                                                                                        
 - $K_c$ : proportional gain
 - $T_i$ : integral time (reset time)
 - $s$ : Laplace variable


In time domain:

$$
u(t)=K_c \left( e(t)+\frac{1}{T_i} \int_0^t e(\tau) d \tau \right)
$$

In [ ]:
# PI Regulator in Interactive Form

s = sympy.Symbol("s")
K_c, T_i = sympy.symbols("K_c, T_i", positive=True)

G_c = K_c * (T_i * s + 1) / (T_i * s)
G_c

In [ ]:
# Time-domain form: u(t) = ?
# G_c(s) = U(s)/E(s), decompose: K_c + K_c/(T_i*s)
# Inverse Laplace of each term, then convolve with e(t):

t, tau = sympy.symbols("t, tau", positive=True)
e = sympy.Function("e")

u_t = K_c * e(t) + (K_c / T_i) * sympy.Integral(e(tau), (tau, 0, t))
sympy.Eq(sympy.Function("u")(t), u_t)

## PID Regulator

## Parallel Form (Åström and Hägglund)

$$
u(t) = K_c \left(e(t)+\frac{1}{T_i} \int_0^t e(\tau) d \tau+T_d \frac{d e(t)}{d t}\right)
$$

## Linear Parametrization Form

$$
u(t)=k_p e(t)+k_i \int_0^t e(\tau) d \tau+k_d \frac{d e(t)}{d t},
$$

## Interactive (Series) Form

Laplace domain:

$$
U(s) = G_c(s) E(s)
$$

$$
G_c(s) = K_c \frac{(T_i s + 1)(T_d s + 1)}{T_i s \,(T_f s + 1)}
$$

Time domain:

$$
T_f \dot{u}(t) + u(t) = K_c \left[ T_d \frac{d e(t)}{d t} + \left(1 + \frac{T_d}{T_i}\right) e(t) + \frac{1}{T_i} \int_0^t e(\tau)\, d\tau \right]
$$

where:
 - $K_c$ : proportional gain
 - $T_i$ : integral time constant (reset time)
 - $T_d$ : derivative time constant
 - $T_f$ : derivative filter time constant
 - $s$ : Laplace variable

In [ ]:
# PID Controller in Interactive Form

T_d, T_f = sympy.symbols("T_d, T_f", positive=True)

G_c_pid = K_c * (T_i * s + 1) * (T_d * s + 1) / (T_i * s * (T_f * s + 1))
G_c_pid

In [ ]:
# Time-domain form (implicit ODE for u(t)):
# Cross-multiply: U(s)*T_i*s*(T_f*s+1) = K_c*(T_i*s+1)*(T_d*s+1)*E(s)
# Divide by T_i*s: (T_f*s+1)*U(s) = K_c*[T_d*s + (1+T_d/T_i) + 1/(T_i*s)]*E(s)
# Inverse Laplace:

u = sympy.Function("u")

lhs = T_f * u(t).diff(t) + u(t)
rhs = K_c * (
    T_d * e(t).diff(t)
    + (1 + T_d / T_i) * e(t)
    + sympy.Integral(e(tau), (tau, 0, t)) / T_i
)
sympy.Eq(lhs, rhs)

### Converting Interactive Form Parameters to Parallel Form

Setting $T_f = 0$ and expanding:

$$
G_c(s) = K_c \frac{(T_i s + 1)(T_d s + 1)}{T_i s}
= K_c T_d \, s + K_c\!\left(1 + \frac{T_d}{T_i}\right) + \frac{K_c}{T_i s}
$$

Matching coefficients with $k_d s + k_p + k_i / s$:

$$
k_p = K_c\!\left(\frac{T_i + T_d}{T_i}\right), \qquad k_i = \frac{K_c}{T_i}, \qquad k_d = K_c T_d
$$

In [ ]:
# Derive parallel parameters by expanding G_c_pid with T_f=0
G_c_ideal = G_c_pid.subs(T_f, 0)
expanded = sympy.apart(G_c_ideal, s)  # partial fractions in s

k_p, k_i, k_d = sympy.symbols("k_p k_i k_d")

def pid_interactive_to_linear_parallel(K_c, T_i, T_d=0.0):
    k_p = K_c * (T_i + T_d) / T_i
    k_i = K_c / T_i
    k_d = K_c * T_d
    return k_p, k_i, k_d

k_p, k_i, k_d = pid_interactive_to_linear_parallel(K_c, T_i, T_d)

for name, expr in [("k_p", k_p), ("k_i", k_i), ("k_d", k_d)]:
    display(sympy.Eq(sympy.Symbol(name), expr))

## Setpoint Filter

First order

$$
Y_f(s)=\frac{1}{s T_f+1} Y(s)
$$

Second order

$$
Y_f(s)=\frac{1}{\left(s T_f+1\right)^2} Y(s)
$$

## Test Case


In [ ]:
# Example
K_p = 2.0
T_1 = 2.0
T_2 = 3.0
T_0 = -1.0

G_p = con.tf([K_p * T_0, K_p], np.convolve([T_1, 1], [T_2, 1]))

print(G_p)

In [ ]:
con.pzmap(G_p)
plt.show()

In [ ]:
# Time vector and step input starting at t=1
t = np.linspace(0, 20, 201)
u = np.where(t >= 1, 1.0, 0.0)

response = con.forced_response(G_p, T=t, U=u)
y = response.y[0]

data = pd.DataFrame(
    {
        "input": u,
        "output": y
    },
    index=pd.Index(t, name="Time (t)")
)

plot_info = {
    "Step Response": {
        'Input $u(t)$': {
            'var_name': 'input', 'kind': 'step', 'linestyle': '--', 'color': 'k'
        },
        'Output $y(t)$': {'var_name': 'output'}
    }
}

fig, axes = make_tsplots(data, plot_info=plot_info)
plt.tight_layout()
plt.show()

In [ ]:
# Simplified design rule
K_c = 1.0 / K_p
T_i = T_1
T_d = T_2

if T_0 > 0.0:
    T_f = T_0
else:
    T_f = T_d / 5

K_c, T_i, T_d, T_f

In [ ]:
G_c = con.tf(
    K_c * np.convolve([T_i, 1], [T_d, 1]),
    np.convolve([T_i, 0], [T_f, 1])
)
print(G_c)

In [ ]:
# Closed-loop simulation: PID regulator + continuous-time plant (no delay)

# Closed-loop transfer function (unity negative feedback)
G_cl = con.feedback(G_c * G_p)
print("Closed-loop poles:", G_cl.poles().round(2))

# Step setpoint starting at t=1
t = np.linspace(0, 25, 1001)
r = np.where(t >= 1, 1.0, 0.0)

# Compute system output response
response = con.forced_response(G_cl, T=t, U=r)
y = response.y[0]

# Compute control signal u(t) = G_c * e(t), e = r - y
e = r - y
u_response = con.forced_response(G_c, T=t, U=e)
u = u_response.y[0]

data = pd.DataFrame(
    {
        "reference": r,
        "input": u,
        "output": y
    },
    index=pd.Index(t, name="Time (t)")
)

plot_info = {
    "Closed-Loop Step Response": {
        'Reference $r(t)$': {
            'var_name': 'reference', 'kind': 'step', 'linestyle': '--', 'color': 'k'
        },
        'Output $y(t)$': {'var_name': 'output'}
    },
    "Control Input": {
        'Input $u(t)$': {
            'var_name': 'input', 'color': 'C1'
        },
    }
}

fig, axes = make_tsplots(data, plot_info=plot_info)
plt.tight_layout()
plt.show()

In [ ]:
con.pzmap(G_cl)
plt.show()

In [ ]:
G_cl.bode_plot()
plt.tight_layout()
plt.show()

## Compare to Discrete Time PID Regulator

In [ ]:
from python_pid import PIDController

k_p, k_i, k_d = pid_interactive_to_linear_parallel(K_c, T_i, T_d)

ctrl = PIDController(k_p, k_i, k_d, TfTs=1.0)
ctrl

